# M3 — KV Cache và profiling
Gắn Kaggle Output của M2 làm Input để notebook tìm `runs/dense/best.pt`. Benchmark chỉ chạy context length không vượt `block_size` của checkpoint. First-token latency gồm prefill + dự đoán token đầu tiên.


In [ ]:
REPO_URL = "https://github.com/shinbkb/mini-gpt.git"
COMMIT_SHA = "REPLACE_WITH_FULL_M3_COMMIT_SHA"
CHECKPOINT = None  # Optional explicit path; otherwise search /kaggle/input/**/runs/dense/best.pt
CONTEXTS = [128, 512, 1024, 2048]
NEW_TOKENS = 64
WARMUP = 2
REPEATS = 5
PRECISION = "fp16"  # Match the checkpoint's intended inference precision
DEVICE = "auto"


In [ ]:
from pathlib import Path
import os, re, subprocess, sys
root = Path('/kaggle/working/mini-gpt-m3')
assert re.fullmatch(r'[0-9a-fA-F]{40}', COMMIT_SHA), 'Set full M3 commit SHA after pushing it.'
subprocess.run(['git','clone',REPO_URL,str(root)], check=True)
subprocess.run(['git','fetch','origin',COMMIT_SHA], cwd=root, check=True)
subprocess.run(['git','checkout','--detach',COMMIT_SHA], cwd=root, check=True)
assert subprocess.check_output(['git','rev-parse','HEAD'], cwd=root, text=True).strip().lower() == COMMIT_SHA.lower()
os.chdir(root)
subprocess.run([sys.executable,'-m','pip','install','-r','requirements.txt'], check=True)
if CHECKPOINT is None:
    matches = list(Path('/kaggle/input').rglob('best.pt'))
    matches = [p for p in matches if 'dense' in p.parts]
    if not matches: raise FileNotFoundError('Attach M2 Notebook Output as Kaggle Input; expected runs/dense/best.pt')
    CHECKPOINT = str(matches[0])
print('Checkpoint:', CHECKPOINT)


In [ ]:
subprocess.run([sys.executable,'-m','pytest','-q'], check=True)


In [ ]:
subprocess.run([sys.executable,'-m','experiments.benchmark_kv',
                '--checkpoint',CHECKPOINT,'--contexts',*map(str,CONTEXTS),
                '--new-tokens',str(NEW_TOKENS),'--warmup',str(WARMUP),
                '--repeats',str(REPEATS),'--precision',PRECISION,
                '--device',DEVICE,'--output','reports/benchmark_kv.json'], check=True)
print('Output: reports/benchmark_kv.json, .csv, .svg')
